In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
## These are the only imports that are going to be used
import os
import math
import random
random.seed(42)


In [2]:
## Downloading a dataset that is a list of strings

if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"num docs: {len(docs)}")

num docs: 32033


In [3]:
## Definfining the tokenizer that converts strings to sequences
uchars = sorted(set(''.join(docs))) # unique characters in the dataset become token ids 0..n-1
BOS = len(uchars) # token id for a special Beginning of Sequence (BOS) token
vocab_size = len(uchars) + 1 # total number of unique tokens, +1 is for BOS
print(f"vocab size: {vocab_size}")

vocab size: 27


In [4]:
uchars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [5]:
# Autograd to automatically calculate gradients for each value after every operation
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads') # Python optimization for memory usage

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # scalar value of this node calculated during forward pass
        self.grad = 0                   # derivative of the loss w.r.t. this node, calculated in backward pass
        self._children = children       # children of this node in the computation graph
        self._local_grads = local_grads # local derivative of this node w.r.t. its children

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    # 
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad


In [6]:
## Hyperparameters
n_layer = 1     # depth of the transformer neural network (number of layers)
n_embd = 16     # embedding dimension
block_size = 16 # maximum context length of the attention window (note: the longest name is 15 characters)
n_head = 4      # number of attention heads
head_dim = n_embd // n_head # derived dimension of each head
# matrix initialization using a gaussian distribution
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
# state dictionary for all learnable weights
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd) # query matrix
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd) # key matrix
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd) # value matrix
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd) # output projection
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd) # fully connected layer
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd) # fully connected laer 2
params = [p for mat in state_dict.values() for row in mat for p in row] # flatten params into a single list[Value]
print(f"num params: {len(params)}")

num params: 4192


In [7]:
## Define functions that would be used in the model architecture

# Feed forward function
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

# Softmax activation function
def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

# Normalize vector magnitude function
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

In [8]:
## Now for the GPT2 architecture

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # token embedding
    pos_emb = state_dict['wpe'][pos_id] # position embedding
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # joint token and position embedding
    x = rmsnorm(x) # note: not redundant due to backward pass via the residual connection

    for li in range(n_layer):
        # 1) Multi-head Attention block
        x_residual = x # residual to be added eventually
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])# query wei
        k = linear(x, state_dict[f'layer{li}.attn_wk']) # key weights
        v = linear(x, state_dict[f'layer{li}.attn_wv']) # value weights
        keys[li].append(k)
        values[li].append(v)
        x_attn = [] # to store attention scores
        for h in range(n_head):
            hs = h * head_dim 
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))] # queries multiplited with keys
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)] # q*k now multiplies the values
            x_attn.extend(head_out) # stor the attention scores
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo']) # mmultiply with the weights for the output 
        x = [a + b for a, b in zip(x, x_residual)] # adding the residuals
        # 2) MLP block
        x_residual = x # another residual
        x = rmsnorm(x) # Normalize
        x = linear(x, state_dict[f'layer{li}.mlp_fc1']) # fully connected layer 1
        x = [xi.relu() for xi in x] # activation funciton
        x = linear(x, state_dict[f'layer{li}.mlp_fc2']) # fully connected layer 2
        x = [a + b for a, b in zip(x, x_residual)] # add the residuals

    logits = linear(x, state_dict['lm_head'])
    return logits

In [9]:
## Initialize Adam optimizer and buffers

learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # first moment buffer
v = [0.0] * len(params) # second moment buffer


In [10]:
## The training loop

# Repeat in sequence
num_steps = 500 # number of training steps
for step in range(num_steps):

    # Take single document, tokenize it, surround it with BOS special token on both sides
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward the token sequence through the model, building up the computation graph all the way to the loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1] # the input and target
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits) # softmax of the output
        loss_t = -probs[target_id].log() # calculate the loss
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # final average loss over the document sequence. May yours be low.

    # Backward the loss, calculating the gradients with respect to all model parameters
    loss.backward()

    # Adam optimizer update: update the model parameters based on the corresponding gradients
    lr_t = learning_rate * (1 - step / num_steps) # linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end='\r')


In [11]:
# Inference: may the model babble back to us
temperature = 0.5 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS # set input token to Begining of sequence 
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits]) # apply temperature scaling for randomness
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id]) # decode output
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")


--- inference (new, hallucinated names) ---
sample  1: kiati
sample  2: amiti
sample  3: kaen
sample  4: jaren
sample  5: mamae
sample  6: karie
sample  7: toran
sample  8: amels
sample  9: sann
sample 10: keene
sample 11: kydia
sample 12: kovie
sample 13: emenan
sample 14: ahasto
sample 15: earan
sample 16: lelie
sample 17: kank
sample 18: lara
sample 19: alela
sample 20: amore


In [12]:
## setting temperature to 1

# Inference: may the model babble back to us
temperature = 1 # in (0, 1], control the "creativity" of generated text, low to high
print("\n--- inference (new, hallucinated names) ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS # set input token to Begining of sequence 
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits]) # apply temperature scaling for randomness
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id]) # decode output
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")


--- inference (new, hallucinated names) ---
sample  1: leinon
sample  2: nlttolo
sample  3: gatucea
sample  4: eiura
sample  5: jahenl
sample  6: katcen
sample  7: abhaea
sample  8: hanls
sample  9: e
sample 10: aewss
sample 11: ianbx
sample 12: ahran
sample 13: loss
sample 14: terom
sample 15: vynya
sample 16: sema
sample 17: jazlant
sample 18: joeto
sample 19: moree
sample 20: amaben


In [13]:
## Architecture plot

# plot_microgpt_architecture.py
from graphviz import Digraph

def build_microgpt_diagram():
    dot = Digraph("microGPT", format="png")
    dot.attr(rankdir="LR", fontsize="12")

    # Styles
    input_style = {"shape": "oval", "style": "filled", "fillcolor": "#E3F2FD"}
    block_style = {"shape": "box", "style": "rounded,filled", "fillcolor": "#E8F5E9"}
    norm_style = {"shape": "box", "style": "rounded,filled", "fillcolor": "#FFF3E0"}
    output_style = {"shape": "oval", "style": "filled", "fillcolor": "#FCE4EC"}

    # Nodes
    dot.node("text", "Raw Text", **input_style)
    dot.node("tokenizer", "Character Tokenizer")
    dot.node("tok_emb", "Token Embedding (wte)", **block_style)
    dot.node("pos_emb", "Position Embedding (wpe)", **block_style)

    dot.node("add_emb", "Add Embeddings (+)")
    dot.node("rms1", "RMSNorm", **norm_style)

    dot.node("attn", "Causal Self-Attention\n(Q,K,V projections)", **block_style)
    dot.node("res1", "Residual Add")

    dot.node("rms2", "RMSNorm", **norm_style)
    dot.node("mlp", "MLP Block\n(square ReLU)", **block_style)
    dot.node("res2", "Residual Add")

    dot.node("lm_head", "Linear LM Head", **block_style)
    dot.node("softmax", "Softmax")
    dot.node("output", "Next Token", **output_style)

    # Connections
    dot.edge("text", "tokenizer")
    dot.edge("tokenizer", "tok_emb")

    dot.edge("tok_emb", "add_emb")
    dot.edge("pos_emb", "add_emb")

    dot.edge("add_emb", "rms1")
    dot.edge("rms1", "attn")
    dot.edge("attn", "res1")

    dot.edge("res1", "rms2")
    dot.edge("rms2", "mlp")
    dot.edge("mlp", "res2")

    dot.edge("res2", "lm_head")
    dot.edge("lm_head", "softmax")
    dot.edge("softmax", "output")

    return dot


if __name__ == "__main__":
    diagram = build_microgpt_diagram()
    diagram.render("microgpt_architecture", view=True)


Error: no "view" mailcap rules found for type "image/png"
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'microgpt_architecture.png'
